# PercentileSelector

The ANOVAFSelector ranking with a cutoff given as a share of the features
instead of a count. `percentile=2` keeps the top 2%.

- **Cost:** one F-test per feature.
- **Strength:** the same setting means the same share on datasets of any width.
- **Weakness:** the F-test's blind spots, and the count grows with the input
  width even when the signal does not.
- **Modes:** `task='classification'` or `'regression'`. Wraps scikit-learn's
  `SelectPercentile`.

In [1]:
import numpy as np
from sklearn.datasets import make_classification

from featsel.selectors import PercentileSelector

# 500 features, the first 10 informative (shuffle=False keeps them first).
X, y = make_classification(n_samples=300, n_features=500, n_informative=10,
                           n_redundant=0, shuffle=False, random_state=0)
truth = set(range(10))

## A share, not a count

In [2]:
chosen = set(PercentileSelector(percentile=2).fit(X, y).get_support(indices=True))
print(f'top 2% of 500 = {len(chosen)} features, {len(chosen & truth)} of them informative')

top 2% of 500 = 10 features, 5 of them informative


## Where it fails

Add noise features and the same percentile keeps more features, though the
signal has not changed. The extra ones are noise.

In [3]:
rng = np.random.default_rng(0)
for n_noise in (0, 500, 4500):
    X_wide = np.column_stack([X, rng.normal(size=(len(y), n_noise))])
    chosen = set(PercentileSelector(percentile=2).fit(X_wide, y).get_support(indices=True))
    print(f'{X_wide.shape[1]:5d} features: kept {len(chosen):3d}, '
          f'{len(chosen & truth)} informative')

  500 features: kept  10, 5 informative
 1000 features: kept  20, 5 informative
 5000 features: kept 100, 5 informative
